In [7]:
from pathlib import Path
import pandas as pd

In [8]:
# --- config -------------------------------------------------------------
_CANDIDATES = [Path("../Results"), Path("Results"), Path("../../Results")]
RESULTS_DIR = next((p.resolve() for p in _CANDIDATES if p.is_dir()), None)
if RESULTS_DIR is None:
    raise FileNotFoundError("No Results directory found. Looked in: "
                            + ", ".join(str(p.resolve()) for p in _CANDIDATES))

ARM_ORDER = ["A_full", "B_no_ratio", "C_no_spx", "D_final"]
METRICS = ["AUROC", "Balanced_Accuracy", "Macro_F1",
           "Asthma_Sensitivity", "COPD_Sensitivity", "Threshold"]
MODEL = "XGBoost (regularized)"

In [9]:
# TABLE III
# Four ARM Ablation per model || Showing mean AUROC
df_ablation = pd.read_csv(RESULTS_DIR / "ablation_four_arm.csv")
df_ablation.head()

,Model,A_full_mean,A_full_sd,B_no_ratio_mean,B_no_ratio_sd,C_no_spx_mean,C_no_spx_sd,D_final_mean,D_final_sd,ratio_effect,total_spx_effect,race_effect,A_minus_D
0,Extra Trees,0.911408,0.027796,0.853104,0.036596,0.797678,0.041947,0.779601,0.042858,0.058305,0.113731,0.018077,0.131808
1,Logistic Regression,0.926670,0.018586,0.897143,0.023893,0.809009,0.037005,0.794141,0.037848,0.029528,0.117661,0.014869,0.132529
2,Random Forest,0.959690,0.011629,0.915969,0.020980,0.817125,0.038039,0.803541,0.038230,0.043721,0.142565,0.013584,0.156149
3,SVM,0.900891,0.173354,0.905751,0.026471,0.767550,0.175641,0.743438,0.179848,-0.004860,0.133341,0.024112,0.157452
4,XGBoost (regularized),0.959307,0.012206,0.926858,0.017484,0.818261,0.037898,0.803757,0.038434,0.032449,0.141047,0.014504,0.155550


In [10]:
# TABLE IV 
# --- per-arm metrics for one model --------------------------------------
rows = {}
for arm in ARM_ORDER:
    f = pd.read_csv(RESULTS_DIR / f"nested_cv_folds_{arm}.csv")
    g = f[f["Model"] == MODEL]
    assert len(g) == 50, f"{arm}: expected 50 folds for {MODEL}, got {len(g)}"
    rows[arm] = {m: f"{g[m].mean():.4f} ± {g[m].std(ddof=1):.4f}" for m in METRICS}

metrics_tbl = pd.DataFrame(rows).loc[METRICS]
print(f"=== {MODEL}: all metrics by arm (mean ± SD over 50 folds) ===")
print(metrics_tbl.to_string())
metrics_tbl.to_csv(RESULTS_DIR / "metrics_by_arm_xgboost.csv")


=== XGBoost (regularized): all metrics by arm (mean ± SD over 50 folds) ===
                             A_full       B_no_ratio         C_no_spx          D_final
AUROC               0.9593 ± 0.0122  0.9269 ± 0.0175  0.8183 ± 0.0379  0.8038 ± 0.0384
Balanced_Accuracy   0.9078 ± 0.0180  0.8320 ± 0.0339  0.7315 ± 0.0406  0.7291 ± 0.0380
Macro_F1            0.8082 ± 0.0320  0.7260 ± 0.0526  0.6412 ± 0.0615  0.6374 ± 0.0497
Asthma_Sensitivity  0.8700 ± 0.0367  0.8085 ± 0.0803  0.7557 ± 0.1128  0.7532 ± 0.0888
COPD_Sensitivity    0.9455 ± 0.0450  0.8555 ± 0.1021  0.7072 ± 0.1231  0.7051 ± 0.1039
Threshold           0.5344 ± 0.1790  0.6639 ± 0.1432  0.5459 ± 0.1179  0.5355 ± 0.0862


In [11]:
# --- full summary across every model and arm ----------------------------
def summarise(fold_df):
    out = []
    for model, g in fold_df.groupby("Model"):
        row = {"Model": model, "N_folds": len(g)}
        for m in METRICS:
            row[f"{m}_mean"] = g[m].mean()
            row[f"{m}_sd"] = g[m].std(ddof=1)
        out.append(row)
    return pd.DataFrame(out)

full = pd.concat(
    [summarise(pd.read_csv(RESULTS_DIR / f"nested_cv_folds_{a}.csv")).assign(Arm=a)
     for a in ARM_ORDER],
    ignore_index=True)
full.to_csv(RESULTS_DIR / "arm_summary_all.csv", index=False)
print(f"\nWrote arm_summary_all.csv ({len(full)} rows)")


Wrote arm_summary_all.csv (20 rows)


In [ ]:
# TABLE V
# Paired Arm Deltas for each model
df_delta = pd.read_csv(RESULTS_DIR / "paired_arm_deltas.csv")
df_delta.head()

,Comparison,Model,delta_mean,boot_lo,boot_hi,n_repeats,wilcoxon_p
0,Total leakage + race (A - D),SVM,0.157452,0.080422,0.227807,10,0.009766
1,Total leakage + race (A - D),Random Forest,0.156149,0.150879,0.161860,10,0.001953
2,Total leakage + race (A - D),XGBoost (regularized),0.155550,0.150970,0.159688,10,0.001953
3,Total leakage + race (A - D),Logistic Regression,0.132529,0.130248,0.135019,10,0.001953
4,Total leakage + race (A - D),Extra Trees,0.131808,0.127367,0.136647,10,0.001953
